In [1]:
"""
강판 결함 데이터: 지수(Index) 변수가 기본 측정값으로 계산된 파생 변수인지 검증
- 계산식으로 직접 값을 만들고, 원본 열과 비교해 일치율과 오차를 출력합니다.
- 원본 값은 소수점 4자리로 반올림되어 있으므로 허용 오차(TOL) 이내면 일치로 봅니다.
"""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1. 데이터 불러오기 (CSV로 변환한 파일 사용)
# ---------------------------------------------------------------
df = pd.read_csv("steel_plates_faults.csv")
# NNA 원본을 바로 쓰려면 아래 두 줄로 대체
# cols = [c.strip() for c in open("Faults27x7_var") if c.strip()]
# df = pd.read_csv("Faults.NNA", sep=r"\s+", header=None, names=cols)

TOL = 1e-3  # 반올림 오차 허용 범위

# ---------------------------------------------------------------
# 2. 계산에 쓸 기본 값
# ---------------------------------------------------------------
dx = df["X_Maximum"] - df["X_Minimum"]          # 결함 가로 길이
dy = df["Y_Maximum"] - df["Y_Minimum"]          # 결함 세로 길이
L = df["Length_of_Conveyer"]                    # 컨베이어(강판 폭) 길이
area = df["Pixels_Areas"]                       # 결함 면적
mean_lum = df["Sum_of_Luminosity"] / area       # 평균 밝기

# ---------------------------------------------------------------
# 3. 계산식으로 파생 변수 직접 생성
# ---------------------------------------------------------------
calc = pd.DataFrame(index=df.index)
calc["Edges_Index"] = 2 * np.minimum(df["X_Minimum"], L - df["X_Maximum"]) / L
calc["Empty_Index"] = 1 - area / (dx * dy)
calc["Square_Index"] = np.minimum(dx, dy) / np.maximum(dx, dy)
calc["Outside_X_Index"] = dx / L
calc["Edges_X_Index"] = dx / df["X_Perimeter"]
calc["Edges_Y_Index"] = dy / df["Y_Perimeter"]
calc["Outside_Global_Index"] = np.select([dy > dx, dy == dx], [1.0, 0.5], default=0.0)
calc["LogOfAreas"] = np.log10(area)
calc["Log_X_Index"] = np.log10(dx)
calc["Log_Y_Index"] = np.log10(dy)
calc["Orientation_Index"] = np.where(dy >= dx, 1 - dx / dy, -(1 - dy / dx))
calc["Luminosity_Index"] = mean_lum / 128 - 1

# ---------------------------------------------------------------
# 4. 원본 값과 비교
# ---------------------------------------------------------------
rows = []
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    rows.append({
        "변수": col,
        "일치 행 수": int((diff <= TOL).sum()),
        "전체 행 수": len(df),
        "일치율(%)": round((diff <= TOL).mean() * 100, 2),
        "최대 오차": round(diff.max(), 6),
        "평균 오차": round(diff.mean(), 6),
        "상관계수": round(np.corrcoef(df[col], calc[col])[0, 1], 6),
    })

result = pd.DataFrame(rows)
print("=== 파생 변수 계산식 검증 결과 ===")
print(result.to_string(index=False))

# ---------------------------------------------------------------
# 5. 일치하지 않는 행이 있으면 예시 출력
# ---------------------------------------------------------------
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    bad = diff > TOL
    if bad.any():
        print(f"\n[{col}] 불일치 {bad.sum()}행 예시")
        print(pd.DataFrame({"원본": df.loc[bad, col],
                            "계산값": calc.loc[bad, col].round(4),
                            "오차": diff[bad].round(4)}).head())

# ---------------------------------------------------------------
# 6. 강종 더미 변수 중복 여부 확인
# ---------------------------------------------------------------
steel_sum = df["TypeOfSteel_A300"] + df["TypeOfSteel_A400"]
print("\n=== 강종 더미 변수 확인 ===")
print("A300 + A400 값 분포:", steel_sum.value_counts().to_dict())
if (steel_sum == 1).all():
    print("→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보이므로 하나만 사용")

# ---------------------------------------------------------------
# 7. 면적 관련 변수 간 상관 (SigmoidOfAreas는 식 미확인, 상관만 확인)
# ---------------------------------------------------------------
print("\n=== 면적 관련 변수 상관계수 (Spearman) ===")
print(df[["Pixels_Areas", "LogOfAreas", "SigmoidOfAreas"]]
      .corr(method="spearman").round(4))

=== 파생 변수 계산식 검증 결과 ===
                  변수  일치 행 수  전체 행 수  일치율(%)    최대 오차    평균 오차  상관계수
         Edges_Index    1941    1941   100.0 0.000055 0.000024   1.0
         Empty_Index    1941    1941   100.0 0.000055 0.000024   1.0
        Square_Index    1941    1941   100.0 0.000055 0.000022   1.0
     Outside_X_Index    1941    1941   100.0 0.000055 0.000025   1.0
       Edges_X_Index    1941    1941   100.0 0.000055 0.000020   1.0
       Edges_Y_Index    1941    1941   100.0 0.000055 0.000014   1.0
Outside_Global_Index    1941    1941   100.0 0.000000 0.000000   1.0
          LogOfAreas    1941    1941   100.0 0.000055 0.000025   1.0
         Log_X_Index    1941    1941   100.0 0.000054 0.000025   1.0
         Log_Y_Index    1941    1941   100.0 0.000054 0.000025   1.0
   Orientation_Index    1941    1941   100.0 0.000055 0.000022   1.0
    Luminosity_Index    1941    1941   100.0 0.000055 0.000026   1.0

=== 강종 더미 변수 확인 ===
A300 + A400 값 분포: {1: 1941}
→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보